# Combining tools with LLMs

## Pipelines

input(pdf) -> implement RAG -> user input prompt -> LLM(assistant) rephrase for better RAG search -> use 

In [2]:
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import List, TypedDict

import data_transformation
from langchain_chroma import Chroma


In [3]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class AgentState(TypedDict):
    question: str # The user's question
    context: List[str] # Retrieved docs 
    relevance: str # "yes" or "no" (from Grader)
    answer: str # Final answer for this specific question
    attempts: int # Loop counter to prevent infinite loops
    rephrased: str # after passing through LLM1

    messages: List[str]


### Create Node

In [5]:

# only implemented once, for embeding and indexing
def implement_RAG(path: str) -> None:

    documents = data_transformation.data_transform(path)

    DB_PATH = "./chroma_db_data"
    Chroma.from_documents(documents=documents, embedding=data_transformation.embed_model, persist_directory=DB_PATH )
    

In [6]:
def LLM1_rephrase(state: AgentState) -> None:
    system_prompt = f"""
    You are an expert Technical Search Query Optimizer. 
    Your goal is to convert a conversational user question into a precise, 
    keyword-heavy search query for a Vector Database. Follow these rules: 
    1. Remove "fluff" words (e.g., "please", "can you tell me", "I want to know"). 
    2. Expand technical acronyms if context is clear (e.g., "ICE" -> "Internal Combustion Engine"). 
    3. Fix potential typos in technical terms. 
    4. Focus on nouns and specific engineering concepts. 
    5. Output ONLY the new query string. No preamble.

    The question is:

    {state["question"]}
    """

    state["rephrased"] = llm.invoke(system_prompt).content

In [15]:
def RAG_search(state: AgentState) -> None:
    state["context"].clear()
    retreive = Chroma(persist_directory="chroma_db_data", embedding_function=data_transformation.embed_model)
    retriever = retreive.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )
    # This method lives inside the 'retriever' variable you just made
    retrieved_docs = retriever.invoke(state["rephrased"])

    for i in retrieved_docs:
        state["context"].append(i.page_content)

In [17]:
def LLM2_decision(state: AgentState) -> None:
    system_prompt = f"""
    You are a grader assessing relevance of a retrieved document to a user question. 
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. 
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.

    question:
    {state["question"]}

    retreived context:
    {"\n".join(state["context"])}
    """ 

    state["relevance"] = llm.invoke(system_prompt).content


### Build Graph